# TAQ 1-minute previous-tick prices (Python / WRDS)

Python port of `src/sas/download_hf_prices.sas`. Same algorithm:
1. PERMNO -> sym_root/sym_suffix mapping via `wrdsapps.taqmclink`.
2. For each day in `[FIRST_DATE, LAST_DATE]`, pull filtered trades from `taqmsec.ctm_YYYYMMDD`.
3. Median by timestamp before 2015-04-01 (microsecond convention pre-CTM update).
4. Previous-tick fill onto a strict 1-minute grid 09:30..16:00.
5. Append PERMNO, save to Parquet.

**Scale warning**: TAQ is heavy. Server-side filtering keeps each day's transfer to ~50-200 MB after WHERE clauses. Pulling **500 names x 1.5 years** through `wrds.Connection` will take 30-60 minutes; the SAS version is much faster because data never leaves WRDS. Default below = **10 tickers x 5 days** (~1 minute) so you can verify the pipeline before scaling.

In [ ]:
import wrds
import pandas as pd
import numpy as np
from pathlib import Path
import time

## 0) WRDS connection

In [ ]:
WRDS_USERNAME = 'aadmberrada'
db = wrds.Connection(wrds_username=WRDS_USERNAME)

## Configuration

Defaults are intentionally small for testing. Scale up after the first successful run.

In [ ]:
FIRST_DATE = '2024-01-02'
LAST_DATE  = '2024-01-08'   # 5 trading days

# Universe: small list of liquid S&P 500 names for testing.
# To use the full S&P 500 universe from the CRSP panel:
#   univ_df = pd.read_parquet('../data/wrds/sp500_constituents_daily.parquet')
#   PERMNO_LIST = univ_df['permno'].unique().tolist()
PERMNO_LIST = [
    14593,   # AAPL  Apple
    10107,   # MSFT  Microsoft
    90319,   # GOOGL Alphabet
    93436,   # TSLA  Tesla
    11703,   # JPM   JP Morgan
    47896,   # KO    Coca-Cola
    18411,   # JNJ   Johnson & Johnson
    14541,   # XOM   ExxonMobil
    21936,   # PFE   Pfizer
    10104,   # ORCL  Oracle
]

OUT_DIR = Path('../data/wrds')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Output filename mirrors SAS naming: HF_PRICES_<FIRST>_<LAST>.parquet
OUT_FILE = OUT_DIR / f"hf_prices_{FIRST_DATE.replace('-','')}_{LAST_DATE.replace('-','')}.parquet"
print(f"Output: {OUT_FILE}")

## 1) PERMNO -> sym_root/sym_suffix mapping (TAQMCLINK)

In [ ]:
permno_tuple = tuple(int(p) for p in PERMNO_LIST)

taq_map = db.raw_sql(f"""
    SELECT DISTINCT permno,
                    sym_root,
                    COALESCE(sym_suffix, '') AS sym_suffix
    FROM wrdsapps.taqmclink
    WHERE permno IN {permno_tuple}
      AND sym_root IS NOT NULL
""")

print(f"Universe PERMNO: {len(permno_tuple):,}")
print(f"(root, suffix) pairs: {len(taq_map):,}")
taq_map.head(15)

## 2) Discover available CTM tables in date range

Mirrors `%build_ctm_list` in the SAS macro. Tables are named `taqmsec.ctm_YYYYMMDD`.

In [ ]:
first_yyyymmdd = FIRST_DATE.replace('-', '')
last_yyyymmdd  = LAST_DATE.replace('-', '')

ctm_tables = db.raw_sql(f"""
    SELECT table_name,
           SUBSTRING(table_name FROM 5 FOR 8) AS yyyymmdd
    FROM information_schema.tables
    WHERE table_schema = 'taqmsec'
      AND table_name ~ '^ctm_[0-9]{{8}}$'
      AND SUBSTRING(table_name FROM 5 FOR 8) BETWEEN '{first_yyyymmdd}' AND '{last_yyyymmdd}'
    ORDER BY table_name
""")

print(f"CTM tables found in range: {len(ctm_tables):,}")
ctm_tables.head()

## 3) Per-day extraction

Mirrors `%process_one_day` in the SAS. Server-side filters:

- RTH only: `09:30:00 <= time_m <= 16:00:00`
- Real trade: `price > 0`
- Major exchanges: `ex IN ('N','P','T','Q','A','D')`
- No correction: `tr_corr = '00'`
- Sale conditions: keep only blank or letters E/F (reject A,B,C,D,G..Z)
- Restrict to mapping universe via `(sym_root, sym_suffix)` filter

Then in pandas: median by timestamp (pre-2015), previous-tick fill onto 1-min grid.

In [ ]:
MEDIAN_CUTOFF = pd.Timestamp('2015-04-01')

# Server-side filter on (sym_root, sym_suffix). Build the IN-list strings.
# (PostgreSQL accepts "WHERE (sym_root, sym_suffix) IN ((...),(...))".)
pairs_sql = ', '.join(
    f"('{r}', '{s}')" for r, s in zip(taq_map['sym_root'], taq_map['sym_suffix'])
)

def fetch_day(yyyymmdd: str) -> pd.DataFrame:
    """Pull filtered trades for one day. Returns DataFrame with columns
    [date, time_m, sym_root, sym_suffix, price]."""
    return db.raw_sql(f"""
        SELECT date,
               time_m,
               sym_root,
               COALESCE(sym_suffix, '') AS sym_suffix,
               price
        FROM taqmsec.ctm_{yyyymmdd}
        WHERE time_m BETWEEN TIME '09:30:00' AND TIME '16:00:00'
          AND price > 0
          AND ex IN ('N','P','T','Q','A','D')
          AND tr_corr = '00'
          AND (tr_scond IS NULL OR tr_scond = '' OR tr_scond !~ '[ABCDGHIJKLMNOPQRSTUVWXYZ]')
          AND (sym_root, COALESCE(sym_suffix, '')) IN ({pairs_sql})
    """, date_cols=['date'])


def previous_tick_minute_grid(trades: pd.DataFrame, day: pd.Timestamp) -> pd.DataFrame:
    """Reindex trades onto a 1-min grid 09:30..16:00 (inclusive),
    forward-filling the last observed price (previous-tick).
    Output columns: [date, time, sym_root, sym_suffix, price]."""
    if trades.empty:
        return trades.iloc[0:0]

    # 391 minutes from 09:30 to 16:00 inclusive
    grid = pd.date_range(
        start=pd.Timestamp(f"{day.date()} 09:30:00"),
        end  =pd.Timestamp(f"{day.date()} 16:00:00"),
        freq='1min',
    )
    grid_t = grid.time

    out_chunks = []
    grouped = trades.groupby(['sym_root', 'sym_suffix'], sort=False)
    for (root, suffix), g in grouped:
        g = g.sort_values('time_m')
        # merge_asof requires both sides sorted by the 'on' key
        left = pd.DataFrame({'time_m': grid_t})
        merged = pd.merge_asof(
            left, g[['time_m', 'price']],
            on='time_m', direction='backward',
        )
        merged.insert(0, 'date',       day.date())
        merged.insert(2, 'sym_root',   root)
        merged.insert(3, 'sym_suffix', suffix)
        out_chunks.append(merged.rename(columns={'time_m': 'time'}))
    return pd.concat(out_chunks, ignore_index=True)


def process_one_day(yyyymmdd: str) -> pd.DataFrame:
    day = pd.to_datetime(yyyymmdd, format='%Y%m%d')
    t0 = time.perf_counter()

    raw = fetch_day(yyyymmdd)

    # Pre-2015 microsecond convention: median by (sym_root, sym_suffix, time_m)
    if day < MEDIAN_CUTOFF and not raw.empty:
        raw = (raw.groupby(['sym_root', 'sym_suffix', 'time_m'], as_index=False)
                  .agg(date=('date', 'first'), price=('price', 'median')))

    minute_grid = previous_tick_minute_grid(raw, day)

    # Append PERMNO + ticker
    if not minute_grid.empty:
        minute_grid = minute_grid.merge(taq_map, on=['sym_root', 'sym_suffix'], how='left')
        minute_grid['ticker'] = (minute_grid['sym_root'].fillna('')
                                 + minute_grid['sym_suffix'].fillna(''))

    elapsed = time.perf_counter() - t0
    print(f"  {yyyymmdd}: {len(raw):>9,} raw trades -> {len(minute_grid):>7,} 1-min rows  [{elapsed:5.1f}s]")
    return minute_grid

## 4) Run the loop and persist

In [ ]:
all_days = []
t0_total = time.perf_counter()
for yyyymmdd in ctm_tables['yyyymmdd']:
    day_df = process_one_day(yyyymmdd)
    if not day_df.empty:
        all_days.append(day_df)
elapsed_total = time.perf_counter() - t0_total

hf_prices = pd.concat(all_days, ignore_index=True) if all_days else pd.DataFrame()
del all_days

# Reorder columns to match SAS output: DATE, TIME, PERMNO, TICKER, PRICE
if not hf_prices.empty:
    hf_prices = hf_prices[['date', 'time', 'permno', 'ticker', 'price']]

hf_prices.to_parquet(OUT_FILE)
print(f"\nTotal: {len(hf_prices):,} rows  ({elapsed_total:.0f}s)")
print(f"Saved -> {OUT_FILE}")

## 5) Sanity checks

In [ ]:
if not hf_prices.empty:
    print(f"Unique days:    {hf_prices['date'].nunique():>4}")
    print(f"Unique PERMNO:  {hf_prices['permno'].nunique():>4}")
    print(f"Unique tickers: {hf_prices['ticker'].nunique():>4}")
    print(f"Price NaN:      {hf_prices['price'].isna().sum():>7,}  ({hf_prices['price'].isna().mean()*100:.2f}% — pre-open with no prior trade)")
    print(f"\nMinute counts per (date, permno) — should be 391 (09:30..16:00 inclusive):")
    cnt = hf_prices.groupby(['date', 'permno']).size()
    print(f"  min={cnt.min()}  median={int(cnt.median())}  max={cnt.max()}")
    print(f"\nFirst 5 rows:")
    print(hf_prices.head().to_string(index=False))
else:
    print("No data extracted.")

## 6) Disconnect

In [ ]:
db.close()